# Session 10. The agent as a researcher

**A human fixes the metric. The agent runs the night shift.**

- checkpoint first: each student demos a prototype fragment
- autoresearch (Karpathy, March 2026): a coding agent runs the experiment loop overnight
- today is mostly a story, plus one small loop that runs
- practice: you write your project's metric document

**Checkpoint first: the prototype demo.**

- each student shows a working fragment of the project, with traces on screen
- a few minutes each, structured feedback from the room

In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  # reads .env once; nothing below opens a file


def chat_model(size: str = "cheap", **kwargs):
    """A model object for the configured provider. A dozen lines, copy them once."""
    name = os.environ[f"MODEL_{size.upper()}"]  # ids live in .env, never in code
    secret = os.environ["LLM_API_KEY"]
    if os.getenv("LLM_REASONING_EFFORT"):  # gpt-5.x: tools need reasoning "none"
        kwargs.setdefault("reasoning_effort", os.environ["LLM_REASONING_EFFORT"])
    # Gemini over OpenAI-compat drops the reasoning signature: turn two 400s
    if os.getenv("LLM_PROVIDER", "openai_compat") == "google_genai":
        return init_chat_model(f"google_genai:{name}", api_key=secret, **kwargs)
    return init_chat_model(
        f"openai:{name}", api_key=secret, base_url=os.environ["LLM_BASE_URL"], **kwargs
    )


print("provider:", os.getenv("LLM_PROVIDER", "openai_compat"),
      "| strong:", os.environ["MODEL_STRONG"])

## The three-file contract

**Three files, three owners. The boundary is the design.**

- `prepare.py` is immutable: data, tokenizer, the metric `val_bpb` — the agent cannot game its own score
- `train.py` is the only file the agent may edit
- `program.md` is the only human interface: research directions, baselines, failure policy
- "you are not programming Python anymore, you are programming the Markdown file" — Karpathy

## The ratchet cycle

**Hypothesis, edit, commit, five minutes of training, measure.**

- an improvement stays; a regression is rolled back with `git reset` — about twelve experiments an hour
- `results.tsv` accumulates every verdict: the agent's memory between experiments
- the author's overnight run: 83 experiments, 15 kept, `val_bpb` 1.000 to 0.975

## The limits

**The ratchet only climbs, and that is a limit too.**

- greedy search never steps back, so it gets stuck in local optima
- a fixed time budget biases against slow-payoff ideas
- dialogue-tuned models avoid risky moves: RLHF timidity
- the creative part — choosing directions, defining "better" — stays human

## The long-horizon harness

**Anthropic's cousin of the same discipline, for multi-session agents.**

- an initializer agent turns the ask into a JSON task list; every task starts unproven

```json
[
  {"task": "reply cites the source document", "passes": false},
  {"task": "empty input returns a helpful message", "passes": false}
]
```

- one task per session; verify before flipping `passes`
- the failure modes it targets: over-ambition and premature completion
- JSON over Markdown: agents overwrite it less

## The same machine in miniature

**A paper-airplane design lab: the same contract in sixty lines.**

- a plain `for` loop over `model.invoke` — session 1's shape, no graph today
- autoresearch ships no orchestration script either: the coding agent itself is the loop

| autoresearch | this toy |
|---|---|
| `prepare.py` freezes `val_bpb` | `flight_meters()`, a cell we agree never to edit |
| `train.py`, the agent's only file | the `DESIGN` dict |
| `program.md` | the `PROGRAM` string |
| commit / `git reset`; `results.tsv` | keep / roll back one assignment; the `results` list |

In [ ]:
def flight_meters(design: dict) -> float:
    """Frozen judge: how far this paper airplane flies, in meters."""
    loss = 0.02 * (design["wing_angle"] - 36) ** 2  # optimum baked into constants
    loss += 4.0 * (design["nose_weight"] - 1.5) ** 2
    loss += 1.5 * (design["fold_depth"] - 4) ** 2
    return round(30.0 - loss, 2)


DESIGN = {"wing_angle": 30.0, "nose_weight": 2.0, "fold_depth": 3.0}  # the train.py analog
BASELINE = flight_meters(DESIGN)
print("baseline flight:", BASELINE, "m")

**The freeze is repo discipline there, a social contract here.**

- one cell we agree never to edit; in autoresearch it is `prepare.py` by rule
- the metric has no knobs the agent can reach: nothing to game
- deterministic on purpose: the same design always flies the same distance

In [ ]:
from langchain_core.tools import tool

# the program.md analog: goal, knobs and policy, no Python in it
PROGRAM = """You tune a paper airplane design so it flies farther.
Editable parameters: wing_angle (degrees), nose_weight (grams), fold_depth (mm).
Propose exactly ONE parameter change per experiment (the new absolute value), then wait for the verdict.
Changes that measure worse get rolled back. All else being equal, simpler is better."""


@tool
def propose_change(param: str, value: float) -> str:
    """Propose one parameter change for the next flight experiment."""
    return "logged"  # the harness applies and measures; the tool is a form


model = chat_model("strong").bind_tools([propose_change])
print(propose_change.name, "| args:", list(propose_change.args))

**One shape, six turns: propose, apply, measure, keep or roll back, log.**

- every prompt resends the current design, best-so-far and the whole log: `train.py` plus `results.tsv` as memory
- the model's only move is one `propose_change` call; the harness does everything else
- rollback is one assignment here, `git reset` there

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

history = [{"role": "system", "content": PROGRAM}]
best, results = BASELINE, []  # the ratchet starts at the measured baseline

for experiment in range(1, 7):  # fixed budget: the toy's overnight
    log = "\n".join(results) or "no experiments yet"
    status = f"Current design: {DESIGN}\nBest so far: {best} m.\nLog:\n{log}"  # train.py + results.tsv
    history.append(HumanMessage(f"{status}\nPropose experiment {experiment}."))
    reply = model.invoke(history)
    history.append(reply)  # the reply object itself, never rebuilt
    if not reply.tool_calls:
        break  # no proposal means the model is done
    call = reply.tool_calls[0]
    for extra in reply.tool_calls[1:]:  # unanswered ids would fail the next invoke
        history.append(
            ToolMessage("ignored: one change per experiment", tool_call_id=extra["id"])
        )
    param, value = call["args"]["param"], call["args"]["value"]
    if param not in DESIGN:
        verdict = f"rejected: {param} is not an editable parameter"
    else:
        candidate = DESIGN | {param: value}  # one change, applied to a copy
        measured = flight_meters(candidate)
        if measured > best:  # strictly better stays: the monotonic ratchet
            DESIGN, best, verdict = candidate, measured, f"kept: {measured} m"
        else:
            verdict = f"rolled back: {measured} m"  # git reset, as one assignment
    results.append(f"{experiment}\t{param}={value}\t{verdict}")
    history.append(ToolMessage(verdict, tool_call_id=call["id"]))

In [ ]:
print("exp\tchange\tverdict")
for row in results:
    print(row)

kept = sum("kept" in row for row in results)
print(f"\n{kept} kept / {len(results)} experiments")
print(f"flight: {BASELINE} -> {best} m")
print("final design:", DESIGN)

**Read it row by row: the run never moved backward.**

- kept rows only ever raise the best — that is the ratchet
- experiment 4 asked for `glue_amount`: rejected, the freeze boundary held
- experiment 5 overshot the optimum and rolled back: greedy search in one line

## What the toy hides

**Everything between sixty lines and the real thing.**

- real training is noisy; this metric is deterministic by construction
- six experiments against 83; one parameter change against editing code
- measuring against one validation set again and again invites overfitting it

## Practice

**Write your project's `program.md`, in your project repository.**

- the document is the interface: someone who never saw your code can run the check
- "frozen" means the agent may run the measuring code, never edit it
- one metric, one number; if you have three metrics, you have none

**Three required sections, nothing else.**

1. the goal, in one sentence
2. ONE fixed metric, with its current baseline value
3. the verification command: one line, runnable exactly as written

**Swap documents with your review pair. Three review questions.**

- can you run the command as written, with nothing added from your head?
- is the result a single number you could compare against tomorrow?
- can the agent influence the measuring code? then it is not frozen

**Required artifacts, committed today.**

- `program.md` in your project repository, with your review pair's verdict noted
- `runs/session-10.md` in the same repository: the metric line, the baseline, the peer verdict, a path to `program.md`
- no trace this session: the artifact is a document

**Stretch, if you finish early.**

- run one real ratchet cycle on your project: change, measure, keep or revert
- append the row to a `results.tsv` beside `program.md`

## The sentence to keep

**The human sets direction and metric. The agent proves quality by measurement.**

- this is a defense criterion: every project answers it with today's document
- session 11 builds the measuring machinery; session 13 accepts evaluation tests built on this metric document

**Further reading.**

- https://github.com/karpathy/autoresearch
- https://www.datacamp.com/tutorial/guide-to-autoresearch
- https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents

## Next time

**Testing agent quality: today's metric document becomes executable evaluations.**